# GPFS (IBM Storage Scale) Multi-Node Deployment on AWS

## Prerequisites

- Python packages: boto3, paramiko, requests
- AWS credentials configured
- SSH key pair in AWS and locally
- Security group allowing SSH and GPFS traffic (ports 22, 1191, 60000-61000)
- GPFS installer zip pre-uploaded to S3
- Sufficient AWS quotas for instances, volumes, and EIPs

**CRITICAL: All resources must be in the same Availability Zone and subnet**

## Configurable Parameters

This notebook supports configurable numbers of GPFS server nodes, client nodes,
and an optional dedicated Kafka node for GPFS watch event streaming.
When `dedicated_kafka_node` is false, Kafka is co-located on gpfs0.

In [ ]:
# Version Information
from __future__ import annotations

NOTEBOOK_VERSION = '1.0'
LAST_UPDATED = '2026-03-18'
GPFS_VERSION = '5.2.3.2'

print(f'Notebook v{NOTEBOOK_VERSION} | GPFS {GPFS_VERSION} | {LAST_UPDATED}')

## Prerequisites Setup

Install Python packages, import shared helpers, and verify AWS credentials via STS.

In [ ]:
# Package Installation
import subprocess
import sys

sys.path.insert(0, '.')

for pkg in ['boto3', 'paramiko', 'requests']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        )

print('✓ All required packages are available')

In [ ]:
# Import Required Packages
import time
from datetime import datetime

import boto3
import paramiko
from botocore.exceptions import ClientError
from botocore.exceptions import NoCredentialsError
from deploy_common import copy_aws_credentials
from deploy_common import install_docker
from deploy_common import install_filebench
from deploy_common import install_gpfs_from_s3
from deploy_common import install_icicle
from deploy_common import launch_instances_parallel
from deploy_common import load_cluster_state
from deploy_common import load_deploy_config
from deploy_common import parse_kafka_topics
from deploy_common import read_github_token
from deploy_common import resolve_maven_version
from deploy_common import resolve_ubuntu_ami
from deploy_common import run_parallel
from deploy_common import save_cluster_state
from deploy_common import setup_logger
from deploy_common import setup_permissions
from deploy_common import setup_root_ssh_and_exchange_keys
from deploy_common import setup_ubuntu
from deploy_common import ssh_exec
from deploy_common import update_ssh_config
from deploy_common import validate_aws_config
from deploy_common import wait_for_ssh

b3 = boto3.__version__
pm = paramiko.__version__
py = sys.version.split()[0]
print(f'boto3={b3} paramiko={pm} python={py}')
print('✓ All packages imported')

In [ ]:
# AWS Credential Verification
try:
    sts = boto3.client('sts', region_name='us-east-1')
    identity = sts.get_caller_identity()
    print(f'✓ Account: {identity["Account"]}')
    print(f'  ARN:     {identity["Arn"]}')
except (NoCredentialsError, ClientError) as e:
    raise RuntimeError(
        f'AWS credentials not configured or invalid: {e}',
    ) from e

## Logger Setup

Configure a file-based logger for the deployment session.

In [ ]:
# Configure File-Based Logger
import logging

LOG_LEVEL = logging.INFO  # Change to logging.DEBUG for verbose output

logger, log_file = setup_logger('deploy-gpfs', LOG_LEVEL)

## Configuration

Configuration is loaded from `deploy-config.yaml` (common + gpfs sections).

In [ ]:
# Configuration Variables
import random
import string

config = load_deploy_config('gpfs')

# --- Cluster ID: load from state file or generate new ---
_state = load_cluster_state('gpfs-state.json', logger=logger)
if _state:
    CLUSTER_ID = _state['cluster_id']
    print(f'Resuming existing cluster: {CLUSTER_ID}')
else:
    CLUSTER_ID = ''.join(random.choices(string.ascii_lowercase, k=5))
    _state = {
        'cluster_id': CLUSTER_ID,
        'created_at': datetime.now().isoformat(),
    }
    save_cluster_state(_state, 'gpfs-state.json', logger=logger)
    print(f'New cluster created: {CLUSTER_ID}')

# --- Convenience aliases ---
FSNAME = config['fs_name']
CLUSTER_NAME = config['cluster_name']
FILESET_NAME = config['fileset_name']
AWS_REGION = config['aws_region']
AWS_AZ = config['aws_az']
KEY_NAME = config['ssh_key_name']
KEY_PATH = config['ssh_key_path']
# The security group implicitly determines the VPC — no vpc_id needed.
SECURITY_GROUP = config['security_group']
PLACEMENT_GROUP = config.get('placement_group', '')
SSH_USER = config['ssh_user']
NUM_SERVERS = config['num_gpfs_servers']
NUM_CLIENTS = config['num_clients']
GPFS_S3_PATH = config['gpfs_s3_path']

logger.info('Configuration loaded')
print(f'Cluster ID: {CLUSTER_ID}')
_kafka_mode = (
    '1 Kafka'
    if config.get('dedicated_kafka_node', True)
    else 'co-located Kafka'
)
print(
    f'Topology: {NUM_SERVERS} GPFS servers'
    f' + {_kafka_mode}'
    f' + {NUM_CLIENTS} clients',
)

In [ ]:
# Configuration Validation
AWS_EIP_DEFAULT_LIMIT = 5

validate_aws_config(KEY_PATH, SECURITY_GROUP, AWS_REGION, AWS_AZ)

assert FSNAME.isalnum(), f"fs_name must be alphanumeric, got: '{FSNAME}'"
assert NUM_SERVERS >= 1, f'num_gpfs_servers must be >= 1, got: {NUM_SERVERS}'
assert NUM_CLIENTS >= 0, f'num_clients must be >= 0, got: {NUM_CLIENTS}'
assert GPFS_S3_PATH.startswith('s3://'), (
    f'gpfs_s3_path must be an S3 URI: {GPFS_S3_PATH}'
)

DEDICATED_KAFKA = config.get('dedicated_kafka_node', True)

total_eips = NUM_SERVERS + (1 if DEDICATED_KAFKA else 0) + NUM_CLIENTS
if total_eips > AWS_EIP_DEFAULT_LIMIT:
    print(
        f'⚠ {total_eips} EIPs needed'
        f' (default AWS limit is {AWS_EIP_DEFAULT_LIMIT})',
    )

print(f'✓ Configuration validated ({total_eips} EIPs needed)')
print(f'  Dedicated Kafka node: {DEDICATED_KAFKA}')

## Dynamic AMI and Maven Resolution

Query AWS for the latest Ubuntu 24.04 AMI in the target region and resolve
the latest Maven version from Maven Central.

In [ ]:
# Dynamic AMI and Maven Lookup
ec2 = boto3.client('ec2', region_name=AWS_REGION)

UBUNTU_AMI_ID, ubuntu_name = resolve_ubuntu_ami(ec2, AWS_REGION)
print(f'✓ Ubuntu AMI: {ubuntu_name}')
print(f'  ID: {UBUNTU_AMI_ID}')
logger.info(f'Resolved AMI: Ubuntu={UBUNTU_AMI_ID}')

MAVEN_VERSION, MAVEN_URL = resolve_maven_version()
print(f'\n✓ Maven version: {MAVEN_VERSION}')
print(f'  URL: {MAVEN_URL}')
logger.info(f'Resolved Maven: {MAVEN_VERSION}')

## Deployment Steps

### Step 1: Launch All Instances in Parallel

Launch GPFS server nodes (each with a data volume), optional dedicated Kafka
node, and client nodes. Allocates EIPs and updates SSH config.

In [ ]:
# Step 1: Launch ALL instances in parallel
start_time = datetime.now()

# Build launch specs for all nodes
all_launch_specs = []

# GPFS server nodes (each with 1 data volume)
for i in range(NUM_SERVERS):
    all_launch_specs.append(
        {
            'name': f'{FSNAME}-{CLUSTER_ID}-gpfs{i}',
            'ami_id': UBUNTU_AMI_ID,
            'instance_type': config['gpfs_server_instance_type'],
            'data_volumes': [('/dev/sdf', config['data_volume_size_gb'])],
            'placement_group': PLACEMENT_GROUP,
        },
    )

# Kafka node (only if dedicated)
if DEDICATED_KAFKA:
    KAFKA_NAME = f'{FSNAME}-{CLUSTER_ID}-kafka'
    all_launch_specs.append(
        {
            'name': KAFKA_NAME,
            'ami_id': UBUNTU_AMI_ID,
            'instance_type': config['kafka_instance_type'],
            'data_volumes': [],
            'placement_group': '',
        },
    )

# Client nodes (no data volume)
for i in range(NUM_CLIENTS):
    all_launch_specs.append(
        {
            'name': f'{FSNAME}-{CLUSTER_ID}-client{i}',
            'ami_id': UBUNTU_AMI_ID,
            'instance_type': config['client_instance_type'],
            'data_volumes': [],
            'placement_group': '',
        },
    )

results = launch_instances_parallel(
    ec2,
    all_launch_specs,
    key_name=KEY_NAME,
    security_group=SECURITY_GROUP,
    availability_zone=AWS_AZ,
    root_volume_size=config['root_volume_size'],
    logger=logger,
)

# --- Extract results into typed node lists ---
gpfs_nodes = []
for i in range(NUM_SERVERS):
    name = f'{FSNAME}-{CLUSTER_ID}-gpfs{i}'
    gpfs_nodes.append(results[name])

if DEDICATED_KAFKA:
    kafka_node = results[KAFKA_NAME]
    KAFKA_EIP = kafka_node['eip']
    KAFKA_PRIVATE_IP = kafka_node['private_ip']
else:
    # Co-located: Kafka runs on gpfs0
    KAFKA_NAME = gpfs_nodes[0]['name']
    kafka_node = gpfs_nodes[0]
    KAFKA_EIP = gpfs_nodes[0]['eip']
    KAFKA_PRIVATE_IP = gpfs_nodes[0]['private_ip']

client_nodes = []
for i in range(NUM_CLIENTS):
    name = f'{FSNAME}-{CLUSTER_ID}-client{i}'
    client_nodes.append(results[name])

# --- Update SSH config ---
ssh_instances = []
for node in gpfs_nodes:
    ssh_instances.append(
        {
            'name': node['name'],
            'ip': node['eip'],
            'user': SSH_USER,
            'key_path': KEY_PATH,
        },
    )
if DEDICATED_KAFKA:
    ssh_instances.append(
        {
            'name': KAFKA_NAME,
            'ip': KAFKA_EIP,
            'user': SSH_USER,
            'key_path': KEY_PATH,
        },
    )
for node in client_nodes:
    ssh_instances.append(
        {
            'name': node['name'],
            'ip': node['eip'],
            'user': SSH_USER,
            'key_path': KEY_PATH,
        },
    )
update_ssh_config(ssh_instances, 'GPFS', logger=logger)

# --- Save state ---
_state['gpfs_nodes'] = [
    {
        'name': n['name'],
        'instance_id': n['instance_id'],
        'eip': n['eip'],
        'private_ip': n['private_ip'],
        'private_dns': n['private_dns'],
        'eip_allocation': n['eip_allocation'],
    }
    for n in gpfs_nodes
]
if DEDICATED_KAFKA:
    _state['kafka_node'] = {
        'name': kafka_node['name'],
        'instance_id': kafka_node['instance_id'],
        'eip': KAFKA_EIP,
        'private_ip': KAFKA_PRIVATE_IP,
        'eip_allocation': kafka_node['eip_allocation'],
    }
else:
    _state['kafka_node'] = {
        'name': KAFKA_NAME,
        'co_located': True,
        'instance_id': gpfs_nodes[0]['instance_id'],
        'eip': KAFKA_EIP,
        'private_ip': KAFKA_PRIVATE_IP,
    }
_state['client_nodes'] = [
    {
        'name': n['name'],
        'instance_id': n['instance_id'],
        'eip': n['eip'],
        'private_ip': n.get('private_ip'),
    }
    for n in client_nodes
]
save_cluster_state(_state, 'gpfs-state.json', logger=logger)

total = NUM_SERVERS + (1 if DEDICATED_KAFKA else 0) + NUM_CLIENTS
logger.info(f'Step 1 completed: {total} instances launched in parallel')

### Step 2: Parallel Node Setup

Three independent pipelines run concurrently:
- **Pipeline A**: GPFS server nodes — Ubuntu setup, GPFS install from S3, kernel module build, /etc/hosts, root SSH key exchange
- **Pipeline B**: Kafka node — Ubuntu setup, Docker install, Kafka docker-compose, topic creation
- **Pipeline C**: Client nodes — Ubuntu setup, Filebench, Icicle

When `DEDICATED_KAFKA` is false, Pipeline B runs after Pipeline A completes (Kafka is co-located on gpfs0).

In [ ]:
# Step 2: Parallel pipelines — GPFS servers, Kafka, clients

MMFS_BIN = '/usr/lpp/mmfs/bin'
GPFS_DEBS_PATH = '/usr/lpp/mmfs/5.2.3.2/gpfs_debs'

FILEBENCH_VERSION = '1.5-alpha3'
FB_BASE = 'https://github.com/filebench/filebench/releases/download'
FILEBENCH_URL = (
    f'{FB_BASE}/{FILEBENCH_VERSION}/filebench-{FILEBENCH_VERSION}.tar.gz'
)


# --- Pipeline A: GPFS Server Setup ---


def gpfs_server_pipeline():
    """Setup all GPFS server nodes: Ubuntu, GPFS install, hosts, SSH keys."""

    def setup_one_server(node):
        eip = node['eip']
        name = node['name']
        ssh = wait_for_ssh(eip, KEY_PATH, SSH_USER)
        if not ssh:
            raise Exception(f'SSH timeout for {name} at {eip}')
        ssh_exec(ssh, eip, f'sudo hostnamectl set-hostname {name}')

        # Ubuntu setup
        logger.info(f'Running Ubuntu setup on {name}')
        if not setup_ubuntu(
            ssh,
            eip,
            MAVEN_VERSION,
            MAVEN_URL,
            SSH_USER,
            AWS_REGION,
            logger=logger,
        ):
            raise RuntimeError(f'Ubuntu setup failed on {name}')

        # GPFS install from S3
        logger.info(f'Installing GPFS on {name}')
        if not install_gpfs_from_s3(ssh, eip, GPFS_S3_PATH, logger=logger):
            raise RuntimeError(f'GPFS install failed on {name}')

        # Install ksh + GPFS deb packages + build kernel module
        logger.info(f'Installing GPFS packages on {name}')
        ssh_exec(ssh, eip, 'sudo apt-get install -y ksh', timeout=60)
        ssh_exec(
            ssh,
            eip,
            f'sudo dpkg -i {GPFS_DEBS_PATH}/*.deb',
            timeout=120,
            logger=logger,
        )
        # gpfs.gui has unmet deps (postgresql, pmcollector) — fix the broken
        # dpkg state so that subsequent apt-get calls (e.g. Docker) succeed.
        ssh_exec(
            ssh,
            eip,
            'sudo apt-get --fix-broken install -y',
            timeout=120,
            logger=logger,
        )
        logger.info(f'Building GPFS kernel module on {name}')
        ec, _, err = ssh_exec(
            ssh,
            eip,
            f'sudo {MMFS_BIN}/mmbuildgpl',
            timeout=300,
            logger=logger,
        )
        if ec != 0:
            raise RuntimeError(f'mmbuildgpl failed on {name}: {err}')

        node['ssh'] = ssh
        return ssh

    # Setup all servers in parallel
    tasks = [(n['name'], lambda n=n: setup_one_server(n)) for n in gpfs_nodes]
    run_parallel(tasks, logger=logger)

    # Configure /etc/hosts on all server nodes
    logger.info('Configuring /etc/hosts on all GPFS servers')
    hosts_entries = '\n'.join(
        f'{n["private_ip"]} {n["private_dns"]}' for n in gpfs_nodes
    )
    for node in gpfs_nodes:
        ssh_exec(
            node['ssh'],
            node['eip'],
            f"echo '{hosts_entries}' | sudo tee -a /etc/hosts",
            logger=logger,
        )

    # Setup root SSH keys across all server nodes
    nodes_for_ssh = [
        (n['ssh'], n['eip'], n['private_dns']) for n in gpfs_nodes
    ]
    setup_root_ssh_and_exchange_keys(nodes_for_ssh, logger=logger)

    logger.info('GPFS server pipeline completed')


# --- Pipeline B: Kafka Setup ---


def kafka_pipeline(ssh=None, eip=None):
    """Setup Kafka node: Ubuntu, Docker, Kafka compose, topics.

    When *ssh* and *eip* are provided the node is already set up (co-located
    mode) and only Docker + Kafka are deployed.
    """
    co_located = ssh is not None
    if not co_located:
        eip = KAFKA_EIP
        ssh = wait_for_ssh(eip, KEY_PATH, SSH_USER)
        if not ssh:
            raise Exception(f'SSH timeout for Kafka at {eip}')
        ssh_exec(ssh, eip, f'sudo hostnamectl set-hostname {KAFKA_NAME}')

        # Ubuntu setup
        logger.info('Running Ubuntu setup on Kafka node')
        setup_ubuntu(
            ssh,
            eip,
            MAVEN_VERSION,
            MAVEN_URL,
            SSH_USER,
            AWS_REGION,
            logger=logger,
        )

    # Docker install
    logger.info('Installing Docker on Kafka node')
    if not install_docker(ssh, eip, SSH_USER, logger=logger):
        raise RuntimeError('Docker install failed on Kafka node')

    # Reconnect SSH to activate docker group
    ssh.close()
    ssh = wait_for_ssh(eip, KEY_PATH, SSH_USER, logger=logger)
    if not ssh:
        raise Exception('Failed to reconnect to Kafka node after Docker')

    # Deploy Kafka via docker-compose
    logger.info('Deploying Kafka via docker-compose')
    _kafka_adv = f'DOCKER://kafka:29092,OUTSIDE://{KAFKA_PRIVATE_IP}:9092'
    _proto_map = 'CONTROLLER:PLAINTEXT,DOCKER:PLAINTEXT,OUTSIDE:PLAINTEXT'
    compose = f"""services:
  kafka:
    image: apache/kafka:3.8.1
    container_name: kafka
    ports:
      - "9092:9092"
    environment:
      KAFKA_PROCESS_ROLES: 'broker,controller'
      KAFKA_NODE_ID: 1
      KAFKA_CONTROLLER_LISTENER_NAMES: 'CONTROLLER'
      KAFKA_CONTROLLER_QUORUM_VOTERS: '1@kafka:9093'
      CLUSTER_ID: 'ci_PQ_sN7Qii-b-Ab9rA'
      KAFKA_LISTENERS: 'DOCKER://:29092,OUTSIDE://:9092,CONTROLLER://:9093'
      KAFKA_ADVERTISED_LISTENERS: '{_kafka_adv}'
      KAFKA_INTER_BROKER_LISTENER_NAME: 'DOCKER'
      KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: '{_proto_map}'
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_MIN_ISR: 1

  kafbat-ui:
    image: ghcr.io/kafbat/kafka-ui:latest
    container_name: kafbat-ui
    ports:
      - "8080:8080"
    depends_on:
      - kafka
    environment:
      KAFKA_CLUSTERS_0_NAME: 'local-kafka'
      KAFKA_CLUSTERS_0_BOOTSTRAPSERVERS: 'kafka:29092'
"""

    sftp = ssh.open_sftp()
    try:
        with sftp.file(f'/home/{SSH_USER}/docker-compose.yaml', 'w') as f:
            f.write(compose)
    finally:
        sftp.close()

    ssh_exec(ssh, eip, 'cd ~ && docker compose up -d', timeout=120)
    logger.info('Waiting for Kafka to be ready')
    time.sleep(15)
    ssh_exec(ssh, eip, 'docker ps', logger=logger)

    # Create topics
    topic_partitions = parse_kafka_topics(config['kafka_topics'])
    for topic, partitions in topic_partitions.items():
        logger.info(f'Creating topic: {topic} ({partitions} partitions)')
        ssh_exec(
            ssh,
            eip,
            f'docker exec kafka /opt/kafka/bin/kafka-topics.sh'
            f' --create --topic {topic}'
            f' --partitions {partitions}'
            f' --replication-factor 1'
            f' --bootstrap-server localhost:9092',
            timeout=30,
        )

    ssh_exec(
        ssh,
        eip,
        'docker exec kafka /opt/kafka/bin/kafka-topics.sh'
        ' --list --bootstrap-server localhost:9092',
        logger=logger,
    )

    kafka_node['ssh'] = ssh
    logger.info('Kafka pipeline completed')
    return ssh


# --- Pipeline C: Client Setup ---


def client_pipeline():
    """Setup all client nodes: Ubuntu, Filebench, Icicle."""
    if not client_nodes:
        return

    github_token = read_github_token()

    def setup_one_client(node):
        eip = node['eip']
        name = node['name']
        ssh = wait_for_ssh(eip, KEY_PATH, SSH_USER)
        if not ssh:
            raise Exception(f'SSH timeout for {name} at {eip}')
        ssh_exec(ssh, eip, f'sudo hostnamectl set-hostname {name}')

        logger.info(f'Running Ubuntu setup on {name}')
        if not setup_ubuntu(
            ssh,
            eip,
            MAVEN_VERSION,
            MAVEN_URL,
            SSH_USER,
            AWS_REGION,
            logger=logger,
        ):
            raise RuntimeError(f'Ubuntu setup failed on {name}')

        logger.info(f'Installing Filebench on {name}')
        if not install_filebench(
            ssh,
            eip,
            FILEBENCH_VERSION,
            FILEBENCH_URL,
            logger=logger,
        ):
            raise RuntimeError(f'Filebench install failed on {name}')

        logger.info(f'Installing Icicle on {name}')
        if not install_icicle(
            ssh,
            eip,
            github_token,
            config['git_username'],
            config['git_repo'],
            config['git_branch'],
            python_version='3.11',
            logger=logger,
        ):
            raise RuntimeError(f'Icicle install failed on {name}')

        node['ssh'] = ssh
        return ssh

    tasks = [
        (n['name'], lambda n=n: setup_one_client(n)) for n in client_nodes
    ]
    run_parallel(tasks, logger=logger)
    logger.info('Client pipeline completed')


# --- Run pipelines ---
if DEDICATED_KAFKA:
    # All three pipelines in parallel
    pipelines = [
        ('gpfs-servers', gpfs_server_pipeline),
        ('kafka', kafka_pipeline),
        ('clients', client_pipeline),
    ]
    run_parallel(pipelines, logger=logger)
else:
    # GPFS + client pipelines first, then Kafka on gpfs0
    pipelines = [
        ('gpfs-servers', gpfs_server_pipeline),
        ('clients', client_pipeline),
    ]
    run_parallel(pipelines, logger=logger)
    # Now run Kafka on gpfs0 (already has ssh from server pipeline)
    kafka_pipeline(ssh=gpfs_nodes[0]['ssh'], eip=gpfs_nodes[0]['eip'])

logger.info('Step 2 completed: all pipelines')

### Step 3: GPFS Cluster Formation

1. Spectrum Scale toolkit setup on Node 0
2. Add additional server nodes to cluster (sequential)
3. Create fileset
4. Add client nodes to cluster (sequential)
5. Install Icicle + permissions on server nodes

In [ ]:
# Step 3: GPFS Cluster Formation

TOOLKIT_PATH = '/usr/lpp/mmfs/5.2.3.2/ansible-toolkit'
node0 = gpfs_nodes[0]
node0_ssh = node0['ssh']
node0_eip = node0['eip']
node0_dns = node0['private_dns']
node0_ip = node0['private_ip']

KAFKA_BROKER_ADDR = f'{KAFKA_PRIVATE_IP}:9092'

# === Phase 1: Toolkit setup on Node 0 ===
logger.info('Phase 1: Spectrum Scale toolkit setup on Node 0')

ssh_exec(node0_ssh, node0_eip, 'pip install cherrypy', logger=logger)

# Setup uses private IP for the install node
ssh_exec(
    node0_ssh,
    node0_eip,
    f'cd {TOOLKIT_PATH} && sudo ./spectrumscale setup -s {node0_ip}',
    timeout=120,
    logger=logger,
)
# Add node: -a=admin, -g=gateway, -q=quorum, -m=manager, -n=NSD server
ssh_exec(
    node0_ssh,
    node0_eip,
    f'cd {TOOLKIT_PATH} && sudo ./spectrumscale node add {node0_dns}'
    ' -a -g -q -m -n',
    timeout=120,
    logger=logger,
)

# List nodes to verify
ssh_exec(
    node0_ssh,
    node0_eip,
    f'cd {TOOLKIT_PATH} && sudo ./spectrumscale node list',
    logger=logger,
)

ssh_exec(node0_ssh, node0_eip, 'sudo wipefs -n /dev/nvme1n1', logger=logger)

# Add NSD using private_dns (must match the node added above)
ec, out, err = ssh_exec(
    node0_ssh,
    node0_eip,
    f'cd {TOOLKIT_PATH} && sudo ./spectrumscale nsd add'
    f' -p {node0_dns} -fs {FSNAME} /dev/nvme1n1',
    timeout=120,
    logger=logger,
)
if ec != 0:
    raise RuntimeError(f'NSD add failed: {err}')

ssh_exec(
    node0_ssh,
    node0_eip,
    f'cd {TOOLKIT_PATH} && sudo ./spectrumscale nsd modify nsd1 -fs {FSNAME}',
    timeout=120,
    logger=logger,
)
ssh_exec(
    node0_ssh,
    node0_eip,
    f'cd {TOOLKIT_PATH} && sudo ./spectrumscale config gpfs -c {CLUSTER_NAME}',
    timeout=120,
    logger=logger,
)
ssh_exec(
    node0_ssh,
    node0_eip,
    f'cd {TOOLKIT_PATH} && sudo ./spectrumscale fileauditlogging enable',
    timeout=120,
    logger=logger,
)
ssh_exec(
    node0_ssh,
    node0_eip,
    f'cd {TOOLKIT_PATH}'
    f' && sudo ./spectrumscale filesystem modify {FSNAME}'
    ' --fileauditloggingenable',
    timeout=120,
    logger=logger,
)
ssh_exec(
    node0_ssh,
    node0_eip,
    f'cd {TOOLKIT_PATH} && sudo ./spectrumscale callhome disable',
    timeout=120,
    logger=logger,
)
ssh_exec(
    node0_ssh,
    node0_eip,
    f'cd {TOOLKIT_PATH}'
    ' && sudo ./spectrumscale config gpfs --ephemeral_port_range 60000-61000',
    timeout=120,
    logger=logger,
)

logger.info('Running precheck')
ec, out, err = ssh_exec(
    node0_ssh,
    node0_eip,
    f'cd {TOOLKIT_PATH} && sudo ./spectrumscale install --precheck',
    timeout=300,
    logger=logger,
)
if ec != 0:
    raise RuntimeError(f'Spectrum Scale precheck failed: {err}')

logger.info('Installing Spectrum Scale on Node 0 (5-10 minutes)')
ec, out, err = ssh_exec(
    node0_ssh,
    node0_eip,
    f'cd {TOOLKIT_PATH} && sudo ./spectrumscale install',
    timeout=900,
    logger=logger,
)
if ec != 0:
    raise RuntimeError(f'Spectrum Scale install failed: {err}')

# Update PATH
ssh_exec(
    node0_ssh,
    node0_eip,
    "echo 'export PATH=/usr/lpp/mmfs/bin:$PATH' | sudo tee -a /root/.bashrc",
)

logger.info('Verifying Node 0 installation')
ec, out, _ = ssh_exec(
    node0_ssh,
    node0_eip,
    f'sudo {MMFS_BIN}/mmgetstate -aL',
    logger=logger,
)
if ec != 0:
    raise RuntimeError('mmgetstate failed — cluster not created')
logger.info(f'Node 0 state: {out.strip()}')

# === Phase 2: Add server nodes 1..N-1 ===
for i in range(1, NUM_SERVERS):
    node = gpfs_nodes[i]
    node_dns = node['private_dns']
    node_eip = node['eip']
    node_ssh = node['ssh']
    logger.info(f'Phase 2: Adding server node {i} ({node_dns})')

    ssh_exec(
        node0_ssh,
        node0_eip,
        f'sudo {MMFS_BIN}/mmaddnode -N {node_dns}',
        timeout=120,
        logger=logger,
    )
    ssh_exec(
        node0_ssh,
        node0_eip,
        f'sudo {MMFS_BIN}/mmchlicense server --accept -N {node_dns}',
        timeout=60,
        logger=logger,
    )
    ssh_exec(
        node0_ssh,
        node0_eip,
        f'sudo {MMFS_BIN}/mmstartup -N {node_dns}',
        timeout=120,
        logger=logger,
    )

    logger.info(f'Waiting for node {i} to become active')
    time.sleep(15)

    ssh_exec(
        node0_ssh,
        node0_eip,
        f'sudo {MMFS_BIN}/mmchnode --quorum -N {node_dns}',
        timeout=60,
        logger=logger,
    )
    ssh_exec(
        node0_ssh,
        node0_eip,
        f'sudo {MMFS_BIN}/mmmount {FSNAME} -N {node_dns}',
        timeout=60,
        logger=logger,
    )

    # Add this node's data volume as NSD
    nsd_num = i + 1
    nsd_name = f'nsd{nsd_num}'
    nsd_stanza = (
        f'%nsd: device=/dev/nvme1n1 nsd={nsd_name}'
        f' servers={node_dns}'
        f' usage=dataAndMetadata pool=system failureGroup={nsd_num}'
    )
    ssh_exec(
        node0_ssh,
        node0_eip,
        f"echo '{nsd_stanza}' | sudo tee /tmp/{nsd_name}.stanza",
        logger=logger,
    )
    ssh_exec(
        node0_ssh,
        node0_eip,
        f'sudo {MMFS_BIN}/mmcrnsd -F /tmp/{nsd_name}.stanza',
        timeout=120,
        logger=logger,
    )

    add_stanza = (
        f'%nsd: nsd={nsd_name}'
        f' usage=dataAndMetadata pool=system failureGroup={nsd_num}'
    )
    ssh_exec(
        node0_ssh,
        node0_eip,
        f"echo '{add_stanza}' | sudo tee /tmp/adddisk_{nsd_name}.stanza",
        logger=logger,
    )
    ssh_exec(
        node0_ssh,
        node0_eip,
        f'sudo {MMFS_BIN}/mmadddisk {FSNAME}'
        f' -F /tmp/adddisk_{nsd_name}.stanza',
        timeout=120,
        logger=logger,
    )

    # Update PATH on this node
    ssh_exec(
        node_ssh,
        node_eip,
        "echo 'export PATH=/usr/lpp/mmfs/bin:$PATH'"
        ' | sudo tee -a /root/.bashrc',
    )

    logger.info(f'Server node {i} added to cluster')

# Verify cluster
ssh_exec(
    node0_ssh,
    node0_eip,
    f'sudo {MMFS_BIN}/mmgetstate -aL',
    logger=logger,
)
ssh_exec(
    node0_ssh,
    node0_eip,
    f'sudo {MMFS_BIN}/mmlsdisk {FSNAME} -L',
    logger=logger,
)

# === Phase 3: Create fileset ===
logger.info('Phase 3: Creating GPFS fileset')
ssh_exec(
    node0_ssh,
    node0_eip,
    f'sudo {MMFS_BIN}/mmcrfileset {FSNAME} {FILESET_NAME}',
    timeout=60,
    logger=logger,
)
ssh_exec(
    node0_ssh,
    node0_eip,
    f'sudo {MMFS_BIN}/mmlinkfileset {FSNAME} {FILESET_NAME}'
    f' -J /ibm/{FSNAME}/{FILESET_NAME}',
    timeout=60,
    logger=logger,
)
ssh_exec(
    node0_ssh,
    node0_eip,
    f'sudo {MMFS_BIN}/mmlsfileset {FSNAME}',
    logger=logger,
)

# === Phase 4: Add client nodes ===
for i, cnode in enumerate(client_nodes):
    c_eip = cnode['eip']
    c_name = cnode['name']
    c_ssh = cnode['ssh']
    c_dns = cnode.get('private_dns', '')
    c_ip = cnode.get('private_ip', '')
    logger.info(f'Phase 4: Adding client node {i} ({c_name})')

    # Get private_dns if missing
    if not c_dns:
        _, c_dns, _ = ssh_exec(c_ssh, c_eip, 'hostname -f')
        c_dns = c_dns.strip()
        cnode['private_dns'] = c_dns

    # Configure /etc/hosts on client
    hosts_entries = '\n'.join(
        f'{n["private_ip"]} {n["private_dns"]}' for n in gpfs_nodes
    )
    hosts_entries += f'\n{c_ip} {c_dns}'
    ssh_exec(
        c_ssh,
        c_eip,
        f"echo '{hosts_entries}' | sudo tee -a /etc/hosts",
        logger=logger,
    )

    # Add client to /etc/hosts on all server nodes
    for snode in gpfs_nodes:
        ssh_exec(
            snode['ssh'],
            snode['eip'],
            f"echo '{c_ip} {c_dns}' | sudo tee -a /etc/hosts",
            logger=logger,
        )

    # Setup root SSH between client and all servers
    all_nodes_for_ssh = [
        (n['ssh'], n['eip'], n['private_dns']) for n in gpfs_nodes
    ] + [(c_ssh, c_eip, c_dns)]
    setup_root_ssh_and_exchange_keys(all_nodes_for_ssh, logger=logger)

    # Install GPFS on client
    logger.info(f'Installing GPFS on client {c_name}')
    copy_aws_credentials(c_ssh, c_eip, SSH_USER, AWS_REGION, logger=logger)
    if not install_gpfs_from_s3(c_ssh, c_eip, GPFS_S3_PATH, logger=logger):
        raise RuntimeError(f'GPFS install failed on {c_name}')

    ssh_exec(c_ssh, c_eip, 'sudo apt-get install -y ksh', timeout=60)
    ssh_exec(
        c_ssh,
        c_eip,
        f'sudo dpkg -i {GPFS_DEBS_PATH}/*.deb',
        timeout=120,
        logger=logger,
    )
    # gpfs.gui has unmet deps — fix broken dpkg state
    ssh_exec(
        c_ssh,
        c_eip,
        'sudo apt-get --fix-broken install -y',
        timeout=120,
        logger=logger,
    )
    ec, _, err = ssh_exec(
        c_ssh,
        c_eip,
        f'sudo {MMFS_BIN}/mmbuildgpl',
        timeout=300,
        logger=logger,
    )
    if ec != 0:
        raise RuntimeError(f'mmbuildgpl failed on {c_name}: {err}')

    # Add to cluster as client
    ssh_exec(
        node0_ssh,
        node0_eip,
        f'sudo {MMFS_BIN}/mmaddnode -N {c_dns}',
        timeout=120,
        logger=logger,
    )
    ssh_exec(
        node0_ssh,
        node0_eip,
        f'sudo {MMFS_BIN}/mmchlicense client --accept -N {c_dns}',
        timeout=60,
        logger=logger,
    )
    ssh_exec(
        node0_ssh,
        node0_eip,
        f'sudo {MMFS_BIN}/mmstartup -N {c_dns}',
        timeout=120,
        logger=logger,
    )
    time.sleep(15)
    ssh_exec(
        node0_ssh,
        node0_eip,
        f'sudo {MMFS_BIN}/mmmount {FSNAME} -N {c_dns}',
        timeout=60,
        logger=logger,
    )

    ssh_exec(
        c_ssh,
        c_eip,
        "echo 'export PATH=/usr/lpp/mmfs/bin:$PATH'"
        ' | sudo tee -a /root/.bashrc',
    )

    logger.info(f'Client {c_name} added to cluster')

# === Phase 5: Icicle + permissions on all nodes ===
logger.info('Phase 5: Installing Icicle and configuring permissions')
github_token = read_github_token()


def setup_server_icicle(node):
    """Install Icicle and Filebench on a GPFS server node."""
    ssh, eip, _ = node['ssh'], node['eip'], node['name']
    install_filebench(
        ssh,
        eip,
        FILEBENCH_VERSION,
        FILEBENCH_URL,
        logger=logger,
    )
    install_icicle(
        ssh,
        eip,
        github_token,
        config['git_username'],
        config['git_repo'],
        config['git_branch'],
        python_version='3.11',
        logger=logger,
    )


icicle_tasks = [
    (n['name'], lambda n=n: setup_server_icicle(n)) for n in gpfs_nodes
]
run_parallel(icicle_tasks, logger=logger)

# Permissions on fileset — groupadd/usermod are local OS commands,
# so setup_permissions must run on every node in the cluster.
all_nodes = gpfs_nodes + client_nodes
for node in all_nodes:
    setup_permissions(
        node['ssh'],
        node['eip'],
        f'/ibm/{FSNAME}/{FILESET_NAME}',
        SSH_USER,
        logger=logger,
    )

# === Phase 6: Verification ===
logger.info('Phase 6: Verification')

ssh_exec(node0_ssh, node0_eip, f'sudo {MMFS_BIN}/mmlscluster', logger=logger)
ssh_exec(
    node0_ssh,
    node0_eip,
    f'sudo {MMFS_BIN}/mmgetstate -aL',
    logger=logger,
)
ssh_exec(
    node0_ssh,
    node0_eip,
    f'sudo {MMFS_BIN}/mmhealth node show',
    logger=logger,
)

# Verify mount on all nodes
all_nodes = gpfs_nodes + client_nodes
for node in all_nodes:
    ssh_exec(
        node['ssh'],
        node['eip'],
        f'sudo {MMFS_BIN}/mmlsmount {FSNAME}',
        logger=logger,
    )

# Cross-node file test
test_path = f'/ibm/{FSNAME}/{FILESET_NAME}/test-verification.txt'
ssh_exec(
    node0_ssh,
    node0_eip,
    f"echo 'Written from Node 0' | sudo tee {test_path}",
    logger=logger,
)
for node in all_nodes[1:]:
    ssh_exec(node['ssh'], node['eip'], f'cat {test_path}', logger=logger)

# Kafka connectivity
for node in gpfs_nodes:
    ec, _, _ = ssh_exec(
        node['ssh'],
        node['eip'],
        f"echo 'test' | timeout 5 bash -c"
        f" 'cat > /dev/tcp/{KAFKA_PRIVATE_IP}/9092'",
        logger=logger,
    )
    if ec == 0:
        logger.info(f'{node["name"]} -> Kafka: OK')

logger.info('Step 3 completed: cluster formation + verification')

### Step 4: GPFS Watches and Kafka Integration

Enable `mmwatch` on the GPFS fileset for each configured Kafka topic, create
a test event, and verify Kafka receives it.

In [ ]:
# Step 4: Configure GPFS watches for Kafka event streaming

logger.info('Configuring GPFS watches')

topic_partitions = parse_kafka_topics(config['kafka_topics'])
kafka_ssh = kafka_node['ssh']

for topic in topic_partitions:
    logger.info(f'Enabling watch for topic: {topic}')
    watch_cmd = (
        f'sudo {MMFS_BIN}/mmwatch {FSNAME} enable'
        f' --fileset {FILESET_NAME}'
        ' --events ALL'
        ' --event-handler kafkasink'
        f' --sink-brokers {KAFKA_BROKER_ADDR}'
        f' --sink-topic {topic}'
        f" --description 'Watch {FILESET_NAME}'"
    )
    ssh_exec(node0_ssh, node0_eip, watch_cmd, timeout=120, logger=logger)

ssh_exec(
    node0_ssh,
    node0_eip,
    f'sudo {MMFS_BIN}/mmwatch all status',
    logger=logger,
)

# Generate a test event and check Kafka
_watch_test = f'/ibm/{FSNAME}/{FILESET_NAME}/watch-test.txt'
ssh_exec(
    node0_ssh,
    node0_eip,
    f"echo 'watch-test' | sudo tee {_watch_test}",
)
time.sleep(5)

first_topic = next(iter(topic_partitions.keys()))
ssh_exec(
    kafka_ssh,
    KAFKA_EIP,
    f'docker exec kafka /opt/kafka/bin/kafka-console-consumer.sh'
    f' --bootstrap-server localhost:9092'
    f' --topic {first_topic}'
    ' --from-beginning --max-messages 5 --timeout-ms 5000 || true',
    logger=logger,
)

logger.info(f'Kafbat UI: http://{KAFKA_EIP}:8080/')
logger.info('Step 4 completed: GPFS watches configured')

## Deployment Summary

Print timing, topology, per-node details, SSH commands, and Kafka verification results.

In [ ]:
# Deployment Summary
end_time = datetime.now()
duration = end_time - start_time

logger.info('=' * 60)
logger.info('DEPLOYMENT COMPLETED SUCCESSFULLY')
logger.info('=' * 60)

logger.info(f'Total deployment time: {duration}')
if DEDICATED_KAFKA:
    logger.info(
        f'Topology: {NUM_SERVERS} GPFS servers'
        f' + 1 Kafka + {NUM_CLIENTS} clients',
    )
else:
    logger.info(
        f'Topology: {NUM_SERVERS} GPFS servers (Kafka co-located on gpfs0)'
        f' + {NUM_CLIENTS} clients',
    )

logger.info('\nGPFS Server Nodes:')
for i, node in enumerate(gpfs_nodes):
    role = 'admin, gateway, quorum, manager' if i == 0 else 'gateway, quorum'
    logger.info(f'  {node["name"]} ({role})')
    logger.info(
        f'    Instance: {node["instance_id"]}'
        f' | Public: {node["eip"]}'
        f' | Private: {node["private_ip"]}',
    )

if DEDICATED_KAFKA:
    logger.info('\nKafka Node:')
    logger.info(f'  {kafka_node["name"]}')
    logger.info(
        f'    Instance: {kafka_node["instance_id"]}'
        f' | Public: {KAFKA_EIP}'
        f' | Private: {KAFKA_PRIVATE_IP}',
    )
else:
    logger.info('\nKafka (co-located on gpfs0):')
    logger.info(
        f'    Public: {KAFKA_EIP} | Private: {KAFKA_PRIVATE_IP}',
    )

for node in client_nodes:
    logger.info(f'\nClient: {node["name"]}')
    logger.info(
        f'    Instance: {node["instance_id"]} | Public: {node["eip"]}',
    )

logger.info('\nGPFS Configuration:')
logger.info(f'  Cluster Name: {CLUSTER_NAME}')
logger.info(f'  Filesystem: {FSNAME}')
logger.info(f'  Fileset: {FILESET_NAME}')
logger.info(f'  Fileset Path: /ibm/{FSNAME}/{FILESET_NAME}')
logger.info(f'  NSDs: {NUM_SERVERS} (one per server node)')

logger.info('\nKafka:')
logger.info(f'  Broker: {KAFKA_BROKER_ADDR}')
topic_partitions = parse_kafka_topics(config['kafka_topics'])
for topic, parts in topic_partitions.items():
    logger.info(f'  Topic: {topic} ({parts} partitions)')

logger.info('\nURLs:')
logger.info(f'  Kafbat UI: http://{KAFKA_EIP}:8080/')

logger.info('\nSSH Access:')
for node in gpfs_nodes:
    logger.info(f'  ssh {node["name"]}')
if DEDICATED_KAFKA:
    logger.info(f'  ssh {KAFKA_NAME}')
for node in client_nodes:
    logger.info(f'  ssh {node["name"]}')

logger.info(f'\nLog file: {log_file}')
logger.info('=' * 60)

# Save final state
_state['completed_at'] = datetime.now().isoformat()
_state['status'] = 'deployed'
save_cluster_state(_state, 'gpfs-state.json', logger=logger)